# 42. Communication Profiling with NCCL | NCCL 通信剖析
**难度：** Hard | **环境：** GPU required | **标签：** `Distributed`, `NCCL`, `Profiling` | **目标人群：** 并行训练与通信工程

先把通信路径、等待时间和重叠关系看清，再做通信 profiling 会更容易定位并行瓶颈，尤其是 all-reduce、同步等待和计算重叠这三类问题。

**关键词：** `nccl`, `all-reduce`, `communication profiling`, `overlap`


## 前置阅读

**导语：** 先看并行策略、通信拓扑和 profiling 方法，再看 NCCL 通信剖析会更容易。

- [27. ZeRO Optimizer Sim | ZeRO 优化器模拟](./27_ZeRO_Optimizer_Sim.ipynb)
- [28. Pipeline Parallelism MicroBatch | Pipeline 并行微批次](./28_Pipeline_Parallelism_MicroBatch.ipynb)
- [29. Tensor Parallelism Sim | Tensor 并行模拟](./29_Tensor_Parallelism_Sim.ipynb)
- [P1: 05. Communication Topologies | 通信拓扑与分布式基石](../01_Hardware_Math_and_Systems/05_Communication_Topologies.md)
- [P1: 13. Profiling and Bottleneck Analysis | 性能分析与瓶颈定位](../01_Hardware_Math_and_Systems/13_Profiling_and_Bottleneck_Analysis.md)
- [P1: 20. NCCL and AllReduce Basics | NCCL 与 AllReduce 基础](../01_Hardware_Math_and_Systems/20_NCCL_and_AllReduce_Basics.md)


### Step 1: 原理与痛点

先明确要观察的通信对象：all-reduce、broadcast、reduce-scatter，还是 rank 间同步等待。通信 profiling 的目标不是只看某个算子慢，而是把通信热点、等待分布和 overlap 状态一起找出来。

### Step 2: 代码实现框架

先搭一个最小的 NCCL profiling 记录框架：采集时间戳、记录通信操作、标注通信与计算是否重叠，再把这些记录整理成表格，最后看谁在阻塞谁。

### Step 3: 关键机制

本页要关注的不是单个通信 API 的名字，而是它们在并行图中的位置、持续时间和重叠关系。只有把通信等待拆出来，才知道瓶颈究竟是带宽、同步、调度，还是某一段 pipeline 把别的阶段卡住了。

### Step 4: 动手实战

请先把最小的通信记录器跑通，再观察通信时间、等待时间和热点分布如何统计。


In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Any


@dataclass
class CommEvent:
    op: str
    start: float
    end: float
    bytes: int
    overlap_with_compute: bool = False

    @property
    def duration(self) -> float:
        return max(self.end - self.start, 0.0)


class NCCLProfilerSim:
    """A tiny NCCL communication profiling simulator.

    It keeps the focus on the teaching loop:
    record -> classify -> overlap -> summarize.
    """

    def __init__(self):
        self.events: List[CommEvent] = []
        self.compute_events: List[Dict[str, float]] = []

    # TODO 1: 记录一个 compute 事件
    def add_compute(self, name: str, start: float, end: float):
        raise NotImplementedError

    # TODO 2: 记录一个通信事件并判断 overlap
    def add_comm(self, op: str, start: float, end: float, bytes: int):
        raise NotImplementedError

    # TODO 3: 判断一个通信区间是否与 compute 重叠
    def _has_overlap(self, start: float, end: float) -> bool:
        raise NotImplementedError

    # TODO 4: 汇总通信事件和 overlap 信息
    def summarize(self) -> Dict[str, Any]:
        raise NotImplementedError

    # TODO 5: 导出时间线
    def timeline(self) -> List[Dict[str, Any]]:
        raise NotImplementedError

🛑 **STOP HERE** 🛑
<br><br><br><br><br><br><br><br><br><br>
> 请先尝试自己完成代码并跑通测试。<br>
> 如果你正在 Colab 中运行，并且遇到困难没有思路，可以向下滚动查看参考答案。
<br><br><br><br><br><br><br><br><br><br>

---

## 参考代码与解析

### 代码

In [ ]:
# TODO：下面是题目区的参考实现。

from dataclasses import dataclass
from typing import List, Dict, Any


@dataclass
class CommEvent:
    op: str
    start: float
    end: float
    bytes: int
    overlap_with_compute: bool = False

    @property
    def duration(self) -> float:
        return max(self.end - self.start, 0.0)


class NCCLProfilerSim:
    """A tiny NCCL communication profiling simulator.

    It keeps the focus on the teaching loop:
    record -> classify -> overlap -> summarize.
    """

    def __init__(self):
        self.events: List[CommEvent] = []
        self.compute_events: List[Dict[str, float]] = []

    def add_compute(self, name: str, start: float, end: float):
        self.compute_events.append({"name": name, "start": start, "end": end})

    def add_comm(self, op: str, start: float, end: float, bytes: int):
        event = CommEvent(op=op, start=start, end=end, bytes=bytes)
        event.overlap_with_compute = self._has_overlap(event.start, event.end)
        self.events.append(event)

    def _has_overlap(self, start: float, end: float) -> bool:
        for c in self.compute_events:
            if not (end <= c["start"] or start >= c["end"]):
                return True
        return False

    def summarize(self) -> Dict[str, Any]:
        total_comm_time = sum(e.duration for e in self.events)
        overlap_time = sum(e.duration for e in self.events if e.overlap_with_compute)
        by_op: Dict[str, Dict[str, float]] = {}
        for e in self.events:
            item = by_op.setdefault(e.op, {"count": 0, "time": 0.0, "bytes": 0})
            item["count"] += 1
            item["time"] += e.duration
            item["bytes"] += e.bytes
        return {
            "num_comm_events": len(self.events),
            "total_comm_time": total_comm_time,
            "overlap_time": overlap_time,
            "overlap_ratio": overlap_time / max(total_comm_time, 1e-8),
            "by_op": by_op,
        }

    def timeline(self) -> List[Dict[str, Any]]:
        return [
            {
                "op": e.op,
                "start": e.start,
                "end": e.end,
                "duration": e.duration,
                "bytes": e.bytes,
                "overlap": e.overlap_with_compute,
            }
            for e in sorted(self.events, key=lambda x: (x.start, x.end, x.op))
        ]


### 解析
- NCCL profiling 的关键是把通信事件、计算事件和 overlap 状态记录清楚。
- 只有先区分通信对象和等待分布，后续才好判断瓶颈到底在通信还是计算。
- 这页的目标不是写一个完整 profiler，而是抓住“记录 -> 分类 -> overlap -> 汇总”这条线。

### 测试

In [ ]:
profiler = NCCLProfilerSim()
profiler.add_compute('forward', 0.0, 2.0)
profiler.add_comm('all_reduce', 1.0, 2.5, 128 * 1024)
profiler.add_comm('broadcast', 2.6, 3.0, 64 * 1024)
profiler.add_compute('backward', 3.0, 5.0)
profiler.add_comm('reduce_scatter', 3.5, 4.3, 96 * 1024)
timeline = profiler.timeline()
summary = profiler.summarize()
assert len(timeline) == 3
assert summary['num_comm_events'] == 3
assert 'all_reduce' in summary['by_op']
print('✅ NCCLProfilerSim 测试通过')